# AirBnb Price Prediction

## Objective
Build a machine learning model to predict Airbnb listing prices in Tokyo based on listing characteristics.

## Problem Type
Regression problem (predicting a continuous variable: price)

In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option('display.float_format', '{:.2f}'.format) 

## Feature Preparation

### Data Cleaning (Loaded from EDA)

In [53]:
df_original = pd.read_csv("../data/listings.csv", encoding='utf-8-sig')
df_clean = df_original.copy()
df_clean = df_clean.dropna(subset=['price'])
df_clean = df_clean.drop(columns=['neighbourhood_group'])
df_clean.fillna({'reviews_per_month': 0}, inplace=True)
df_clean = df_clean.drop(columns=['last_review'])
upper_limit = df_clean['price'].quantile(0.99)
df_clean = df_clean[df_clean['price'] <= upper_limit]
df_clean = df_clean[df_clean['minimum_nights'] <= 30]

### Features

In [54]:
df = df_clean.copy()
df.shape

y = df['price']

features = [
    'room_type',
    'neighbourhood',
    'availability_365',
    'number_of_reviews',
    'minimum_nights'
]

X = df[features]

In [55]:
X = pd.get_dummies(X, drop_first=True)
X.head()

,availability_365,number_of_reviews,minimum_nights,room_type_Hotel room,room_type_Private room,room_type_Shared room,neighbourhood_Akiruno Shi,neighbourhood_Akishima Shi,neighbourhood_Arakawa Ku,neighbourhood_Bunkyo Ku,neighbourhood_Chiyoda Ku,neighbourhood_Chofu Shi,neighbourhood_Chuo Ku,neighbourhood_Edogawa Ku,neighbourhood_Fuchu Shi,neighbourhood_Fussa Shi,neighbourhood_Hachioji Shi,neighbourhood_Hamura Shi,neighbourhood_Higashimurayama Shi,neighbourhood_Higashiyamato Shi,neighbourhood_Hino Shi,neighbourhood_Hinohara Mura,neighbourhood_Inagi Shi,neighbourhood_Itabashi Ku,neighbourhood_Katsushika Ku,neighbourhood_Kita Ku,neighbourhood_Kiyose Shi,neighbourhood_Kodaira Shi,neighbourhood_Koganei Shi,neighbourhood_Kokubunji Shi,neighbourhood_Komae Shi,neighbourhood_Koto Ku,neighbourhood_Kunitachi Shi,neighbourhood_Machida Shi,neighbourhood_Meguro Ku,neighbourhood_Minato Ku,neighbourhood_Mitaka Shi,neighbourhood_Musashino Shi,neighbourhood_Nakano Ku,neighbourhood_Nerima Ku,neighbourhood_Nishitokyo Shi,neighbourhood_Okutama Machi,neighbourhood_Ome Shi,neighbourhood_Ota Ku,neighbourhood_Setagaya Ku,neighbourhood_Shibuya Ku,neighbourhood_Shinagawa Ku,neighbourhood_Shinjuku Ku,neighbourhood_Suginami Ku,neighbourhood_Sumida Ku,neighbourhood_Tachikawa Shi,neighbourhood_Taito Ku,neighbourhood_Tama Shi,neighbourhood_Toshima Ku
0,183,190,3,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
1,76,272,3,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,305,281,5,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
3,80,284,1,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
4,253,150,2,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [56]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((20079, 54), (5020, 54))

Train-test ratio 8/2

In [57]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

Log transform because the data is right-skewed. This helps:
- reducing skew
- makes model stable
- improving prediction

## Data Preparation Summary

- Selected features include room type, neighborhood, availability, number of reviews, and minimum nights based on EDA findings.
- Categorical variables are converted into numerical format using one-hot encoding.
- The dataset was split into training and testing sets using 80/20 ratio.
- Applied log transformation into target variable (price) to reduce skewness and improve model performance.

## First Model (Linear Regression)

### Train Model

In [58]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train_log)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


### Make Predictions

In [59]:
y_pred_log = model.predict(X_test)

In [60]:
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test_log)

Because model was log transformed, we revert it back

### Evaluate Model

In [61]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

mae = mean_absolute_error(y_test_actual, y_pred)
rmse = root_mean_squared_error(y_test_actual, y_pred)

mae, rmse

(8808.625052699785, 14789.759411488772)

### Model Evaluation (Linear Regression)

The linear regression was trained to predict Airbnb listing prices.

The model achieved:
- Mean Absolute Error (MAE) : 8808.63
- Root Mean Squared Error : 14789.76

These results indicate that, on average, the model's predictions deviate from actual prices by approximately ¥8808 on average. The higher RMSE suggests the presence of larger prediction errors for some listings, indicating that the model may struggle to capture complex pricing patterns.

## Improving Model

### Scaling Numerical Features

In [62]:
num_cols = ['availability_365', 'number_of_reviews', 'minimum_nights']

In [63]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

### Removing Weak Features

In [64]:
features_reduced = [
    'room_type',
    'neighbourhood',
    'minimum_nights'
]
X_reduced = df[features_reduced]
X_reduced = pd.get_dummies(X_reduced, drop_first=True)

X_train_reduced, X_test_reduced, y_train, y_test = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42
)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

### Retrain model

In [65]:
model_reduced = LinearRegression()
model_reduced.fit(X_train_reduced, y_train_log)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [66]:
y_pred_log = model_reduced.predict(X_test_reduced)
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test_log)

In [67]:
mae_reduced = mean_absolute_error(y_test_actual, y_pred)
rmse_reduced = root_mean_squared_error(y_test_actual, y_pred)

mae_reduced, rmse_reduced

(8884.339359280595, 14910.017464240478)

### Modified Model Evaluation (Linear Regression)

- MAE becomes 8885 and RMSE becomes 14910m, the results got worse slightly.
- Even if reviews and availability are weak, they are still useful for model.
- Probable sign of underfitting.